In [19]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import mujoco.viewer
from hexapod_env import HexapodEnv

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
N_ENVS = 8
TOTAL_STEPS = 3_000_000
RUN_NAME = "stage3-stroke-air"
# BASE = "models/stage2-tripod"
env = HexapodEnv()

In [21]:
def make_env():
    return Monitor(HexapodEnv())

vec_env = SubprocVecEnv([make_env for _ in range(N_ENVS)])
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
# vec_env = VecNormalize.load(f"{BASE}/vecnormalize.pkl", vec_env)
# vec_env.training = True
# vec_env.norm_reward = True

In [22]:
INFO_KEYS = (
    "forward_vel",
    "side_vel",
    "reward",
    "r_vel",
    "r_gait",
    "r_yaw",
    "r_vy",
    "r_air",
    "pen_roll",
    "pen_pitch",
    "pen_stroke",
    "pen_rate",
)

class InfoCallback(BaseCallback):
    def _on_step(self):
        for info in self.locals["infos"]:
            for k in INFO_KEYS:
                self.logger.record_mean("env/" + k, info[k])
        return True

In [23]:
model = PPO(
    "MlpPolicy",
    vec_env,
    n_steps=1024,
    batch_size=2048,
    n_epochs=5,
    learning_rate=lambda progress: 3e-4 * progress,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    policy_kwargs=dict(
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
        log_std_init=-1.0,
    ),
    tensorboard_log="logs/",
    device="cpu",
    verbose=1,
)

checkpoint = CheckpointCallback(
    save_freq=100_000 // N_ENVS,
    save_path=f"models/{RUN_NAME}/checkpoints",
    save_vecnormalize=True,
)

Using cpu device


In [ ]:
model.learn(total_timesteps=TOTAL_STEPS, callback=[checkpoint, InfoCallback()], tb_log_name=RUN_NAME)

model.save(f"models/{RUN_NAME}/model")
vec_env.save(f"models/{RUN_NAME}/vecnormalize.pkl")
vec_env.close()

Logging to logs/stage3-stroke-air_1
height=0.1129, gravity_z=-0.9999, terminated=False, truncated=True
height=0.1144, gravity_z=-0.9999, terminated=False, truncated=True
height=0.1137, gravity_z=-0.9996, terminated=False, truncated=True
height=0.1141, gravity_z=-0.9999, terminated=False, truncated=True
height=0.1143, gravity_z=-1.0000, terminated=False, truncated=True
height=0.1106, gravity_z=-1.0000, terminated=False, truncated=True
height=0.1163, gravity_z=-0.9999, terminated=False, truncated=True
height=0.1161, gravity_z=-1.0000, terminated=False, truncated=True
----------------------------------
| env/               |           |
|    forward_vel     | 0.000852  |
|    pen_pitch       | 0.000153  |
|    pen_rate        | 4.82      |
|    pen_roll        | 0.000101  |
|    pen_stroke      | 0.00162   |
|    r_air           | -0.0284   |
|    r_gait          | 0.0119    |
|    r_vel           | 0.00568   |
|    r_vy            | 0.971     |
|    r_yaw           | 0.961     |
|    rew